# So sánh Phase 1 và Phase 2 trên test của Phase 1

Notebook này **chỉ đánh giá, không huấn luyện**. Cả hai ViT5-base được nạp lên GPU cùng lúc, sau đó được chấm lần lượt trên đúng cùng một tập test.

Trên Kaggle, chọn **Add Input → Notebook Output** và thêm output của cả notebook phase 1 lẫn phase 2. Cell chọn checkpoint sẽ tự tìm hai thư mục `best`. Nếu tự tìm sai, điền đường dẫn thủ công.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/dungcony/sumarization.git"
current_dir = Path.cwd()
kaggle_project_dir = Path("/kaggle/working/sumarization")

if (current_dir / "src").is_dir() and (current_dir / "configs").is_dir():
    project_dir = current_dir
elif (kaggle_project_dir / "src").is_dir():
    project_dir = kaggle_project_dir
else:
    clone_parent = Path("/kaggle/working") if Path("/kaggle/working").exists() else current_dir
    project_dir = clone_parent / "sumarization"
    if not project_dir.exists():
        print(f"Đang tải source code vào {project_dir} ...", flush=True)
        subprocess.run(["git", "clone", REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"],
    check=True,
)
print(f"✅ Project directory: {Path.cwd()}", flush=True)

In [ ]:
import gc
import json
import math
import time
from glob import glob

import torch
from datasets import concatenate_datasets, load_dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.config import apply_overrides, load_config
from src.data import clean_text
from src.evaluator import build_compute_metrics
from src.model import load_model, load_tokenizer
from src.utils import format_duration, save_json

assert torch.cuda.is_available(), (
    "Không tìm thấy CUDA GPU. Trên Kaggle hãy bật Settings → Accelerator → GPU."
)
print(f"✅ CUDA devices: {torch.cuda.device_count()}", flush=True)
for device_index in range(torch.cuda.device_count()):
    print(f"   GPU {device_index}: {torch.cuda.get_device_name(device_index)}", flush=True)

## 1. Chọn hai model

Để trống hai biến dưới đây để tự tìm. Đường dẫn phải trỏ tới thư mục `best` chứa `config.json`, file trọng số và tokenizer.

In [ ]:
# Ví dụ:
# MANUAL_PHASE_1_PATH = "/kaggle/input/notebooks/.../outputs_phase_1/vit5_base/best"
# MANUAL_PHASE_2_PATH = "/kaggle/input/notebooks/.../outputs/vit5_base/best"
MANUAL_PHASE_1_PATH = ""
MANUAL_PHASE_2_PATH = ""

MODEL_WEIGHT_FILES = (
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
)
TOKENIZER_FILES = ("spiece.model", "tokenizer.json", "tokenizer_config.json")


def validate_checkpoint(path, label):
    path = Path(path).expanduser().resolve()
    missing = []
    if not path.is_dir():
        missing.append("thư mục checkpoint")
    else:
        if not (path / "config.json").is_file():
            missing.append("config.json")
        if not any((path / name).is_file() for name in MODEL_WEIGHT_FILES):
            missing.append("model.safetensors hoặc pytorch_model.bin")
        if not any((path / name).is_file() for name in TOKENIZER_FILES):
            missing.append("tokenizer")
    if missing:
        raise FileNotFoundError(f"{label} không hợp lệ tại {path}; thiếu: {', '.join(missing)}")
    return path


def discover_checkpoints(phase_number):
    phase_tokens = (f"outputs_phase_{phase_number}", f"phase_{phase_number}", f"phase-{phase_number}")
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    candidates = set()
    for root in roots:
        if not root.exists():
            continue
        for config_file in root.rglob("config.json"):
            candidate = config_file.parent.resolve()
            text = str(candidate).casefold()
            if not any(token in text for token in phase_tokens):
                continue
            if any((candidate / name).is_file() for name in MODEL_WEIGHT_FILES):
                candidates.add(candidate)
    return sorted(
        candidates,
        key=lambda path: (path.name.casefold() == "best", "outputs_phase" in str(path).casefold(), str(path)),
        reverse=True,
    )


def choose_checkpoint(manual_path, phase_number):
    label = f"Phase {phase_number}"
    if manual_path.strip():
        return validate_checkpoint(manual_path, label), []
    candidates = discover_checkpoints(phase_number)
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy model {label}. Hãy Add Input output của notebook {label} "
            f"hoặc điền MANUAL_PHASE_{phase_number}_PATH."
        )
    return validate_checkpoint(candidates[0], label), candidates


phase_1_path, phase_1_candidates = choose_checkpoint(MANUAL_PHASE_1_PATH, 1)
phase_2_path, phase_2_candidates = choose_checkpoint(MANUAL_PHASE_2_PATH, 2)

print("=" * 72, flush=True)
print("CHECKPOINTS ĐƯỢC CHỌN", flush=True)
print("=" * 72, flush=True)
print(f"Phase 1: {phase_1_path}", flush=True)
print(f"Phase 2: {phase_2_path}", flush=True)
assert phase_1_path != phase_2_path, "Phase 1 và Phase 2 đang trỏ tới cùng một thư mục."
print("✅ Hai checkpoint khác nhau và có đủ model/tokenizer.", flush=True)

## 2. Nạp duy nhất test Phase 1

Hai model dùng cùng tokenizer và cùng cấu hình sinh để phép so sánh công bằng. Đặt `MAX_TEST_SAMPLES=100` nếu chỉ muốn kiểm tra nhanh; để `None` sẽ chấm toàn bộ test.

In [ ]:
MAX_TEST_SAMPLES = None
EVAL_BATCH_SIZE = 1
GENERATION_NUM_BEAMS = 2

common_config = load_config("configs/vit5_base_phase_1.yaml")
common_config = apply_overrides(common_config, {
    "model.name_or_path": str(phase_1_path),
    "training.per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "generation.num_beams": GENERATION_NUM_BEAMS,
})
tokenizer = load_tokenizer(common_config.model)

# Xác nhận tokenizer phase 2 tương thích với tokenizer phase 1.
phase_2_tokenizer_config = apply_overrides(common_config, {
    "model.name_or_path": str(phase_2_path),
})
phase_2_tokenizer = load_tokenizer(phase_2_tokenizer_config.model)
assert len(tokenizer) == len(phase_2_tokenizer), "Vocabulary hai tokenizer không cùng kích thước."
assert tokenizer.pad_token_id == phase_2_tokenizer.pad_token_id, "pad_token_id không khớp."
assert tokenizer.eos_token_id == phase_2_tokenizer.eos_token_id, "eos_token_id không khớp."
del phase_2_tokenizer

test_files = sorted(Path(path) for path in glob(str(common_config.data.test_file), recursive=True))
if not test_files:
    raise FileNotFoundError(f"Không tìm thấy test phase 1: {common_config.data.test_file}")

parts = []
for suffix in (".csv", ".parquet", ".pq"):
    matching = [str(path) for path in test_files if path.suffix.casefold() == suffix]
    if not matching:
        continue
    file_format = "csv" if suffix == ".csv" else "parquet"
    parts.append(load_dataset(file_format, data_files={"test": matching}, split="test"))
raw_test = parts[0] if len(parts) == 1 else concatenate_datasets(parts)

required_columns = {"article", "summary"}
missing_columns = required_columns - set(raw_test.column_names)
if missing_columns:
    raise ValueError(f"Test phase 1 thiếu cột: {sorted(missing_columns)}")
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test.select(range(min(MAX_TEST_SAMPLES, len(raw_test))))


def tokenize_phase_1_test(examples):
    inputs = [
        common_config.data.source_prefix + clean_text(article)
        for article in examples["article"]
    ]
    targets = [clean_text(summary) for summary in examples["summary"]]
    model_inputs = tokenizer(
        inputs,
        max_length=common_config.data.max_source_length,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=common_config.data.max_target_length,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


phase_1_test = raw_test.map(
    tokenize_phase_1_test,
    batched=True,
    remove_columns=raw_test.column_names,
    desc="Tokenizing Phase 1 test",
)
compute_metrics = build_compute_metrics(tokenizer)
print(f"✅ Phase 1 test: {len(phase_1_test)} samples", flush=True)
print(f"   Source max length: {common_config.data.max_source_length}", flush=True)
print(f"   Target max length: {common_config.data.max_target_length}", flush=True)
print(f"   Generation beams:  {GENERATION_NUM_BEAMS}", flush=True)

In [ ]:
class EvaluationProgressCallback(TrainerCallback):
    def __init__(self, label, total_batches, print_every=50):
        self.label = label
        self.total_batches = max(1, int(total_batches))
        self.print_every = max(1, int(print_every))
        self.completed = 0
        self.started_at = None

    def on_prediction_step(self, args, state, control, **kwargs):
        if self.started_at is None:
            self.started_at = time.time()
        self.completed += 1
        if state.is_world_process_zero and (
                self.completed == 1
                or self.completed % self.print_every == 0
                or self.completed >= self.total_batches
        ):
            elapsed = time.time() - self.started_at
            percent = min(100.0, 100 * self.completed / self.total_batches)
            print(
                f"[{self.label}] batch={self.completed}/{self.total_batches} "
                f"({percent:.1f}%) | elapsed={format_duration(elapsed)}",
                flush=True,
            )


def gpu_memory_text():
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
    return f"GPU 0 RAM={allocated:.2f}/{reserved:.2f} GB (allocated/reserved)"


def load_checkpoint_to_gpu(model_path, display_name):
    model_config = apply_overrides(common_config, {
        "model.name_or_path": str(model_path),
        "generation.num_beams": GENERATION_NUM_BEAMS,
    })
    print(f"Đang nạp {display_name}: {model_path}", flush=True)
    model = load_model(model_config.model, tokenizer, model_config.generation)
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True
    model.to(torch.device("cuda:0"))
    model.eval()
    print(f"✅ Đã nạp {display_name} lên GPU | {gpu_memory_text()}", flush=True)
    return model


def evaluate_preloaded_model(model, model_path, metric_prefix, display_name):
    print("\n" + "=" * 72, flush=True)
    print(f"ĐANG CHẤM {display_name} — MODEL ĐÃ CÓ TRÊN GPU", flush=True)
    print(f"Checkpoint: {model_path}", flush=True)
    print(gpu_memory_text(), flush=True)
    print("=" * 72, flush=True)

    eval_args = Seq2SeqTrainingArguments(
        output_dir=f"/kaggle/working/eval_{metric_prefix}",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        fp16=True,
        predict_with_generate=True,
        generation_max_length=common_config.generation.max_length,
        generation_num_beams=GENERATION_NUM_BEAMS,
        eval_accumulation_steps=1,
        report_to=[],
        disable_tqdm=True,
        dataloader_drop_last=False,
    )
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
    )
    effective_eval_batch = EVAL_BATCH_SIZE * max(1, torch.cuda.device_count())
    total_batches = math.ceil(len(phase_1_test) / effective_eval_batch)
    progress = EvaluationProgressCallback(display_name, total_batches)
    evaluator = Seq2SeqTrainer(
        model=model,
        args=eval_args,
        eval_dataset=phase_1_test,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[progress],
    )

    started_at = time.time()
    results = evaluator.evaluate(metric_key_prefix=metric_prefix)
    results[f"{metric_prefix}_wall_time"] = time.time() - started_at

    del evaluator
    del collator
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ {display_name} hoàn tất; cả hai model vẫn được giữ trên GPU.", flush=True)
    return results

## 3. Chạy lần lượt và so sánh

Cả hai model được nạp lên GPU trước. Evaluation vẫn chạy lần lượt để thu được hai bộ metric độc lập.

In [ ]:
output_dir = Path("/kaggle/working/phase_1_test_comparison")
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 72, flush=True)
print("NẠP CẢ HAI MODEL LÊN GPU", flush=True)
print("=" * 72, flush=True)
phase_1_model = load_checkpoint_to_gpu(phase_1_path, "PHASE 1")
phase_2_model = load_checkpoint_to_gpu(phase_2_path, "PHASE 2")
print(f"✅ CẢ HAI MODEL ĐANG Ở TRÊN GPU | {gpu_memory_text()}", flush=True)

phase_1_results = evaluate_preloaded_model(
    phase_1_model,
    phase_1_path,
    metric_prefix="phase1",
    display_name="PHASE 1",
)
save_json(phase_1_results, output_dir / "phase_1_on_phase_1_test.json")

phase_2_results = evaluate_preloaded_model(
    phase_2_model,
    phase_2_path,
    metric_prefix="phase2",
    display_name="PHASE 2",
)
save_json(phase_2_results, output_dir / "phase_2_on_phase_1_test.json")

comparison = {
    "dataset": "phase_1_test",
    "samples": len(phase_1_test),
    "phase_1_checkpoint": str(phase_1_path),
    "phase_2_checkpoint": str(phase_2_path),
    "generation_num_beams": GENERATION_NUM_BEAMS,
    "metrics": {},
}

print("\n" + "=" * 72, flush=True)
print("SO SÁNH TRÊN TEST PHASE 1", flush=True)
print("=" * 72, flush=True)
print(f"{'Metric':<12} {'Phase 1':>12} {'Phase 2':>12} {'Δ P2-P1':>12}", flush=True)
print("-" * 52, flush=True)

for metric in ("loss", "rouge1", "rouge2", "rougeL"):
    phase_1_key = f"phase1_{metric}"
    phase_2_key = f"phase2_{metric}"
    if phase_1_key not in phase_1_results or phase_2_key not in phase_2_results:
        continue
    phase_1_value = float(phase_1_results[phase_1_key])
    phase_2_value = float(phase_2_results[phase_2_key])
    delta = phase_2_value - phase_1_value
    comparison["metrics"][metric] = {
        "phase_1": phase_1_value,
        "phase_2": phase_2_value,
        "phase_2_minus_phase_1": delta,
        "phase_2_improvement": -delta if metric == "loss" else delta,
        "higher_is_better": metric != "loss",
    }
    print(
        f"{metric:<12} {phase_1_value:>12.4f} "
        f"{phase_2_value:>12.4f} {delta:>+12.4f}",
        flush=True,
    )

save_json(comparison, output_dir / "phase_1_vs_phase_2_on_phase_1_test.json")

rouge_l = comparison["metrics"].get("rougeL")
if rouge_l is not None:
    change = rouge_l["phase_2_improvement"]
    print("\nKẾT LUẬN THEO ROUGE-L TRÊN TEST PHASE 1", flush=True)
    if change > 0:
        print(f"✅ Phase 2 vẫn tốt hơn Phase 1: {change:+.4f} điểm.", flush=True)
    elif change < 0:
        print(
            f"⚠️ Phase 2 giảm {abs(change):.4f} điểm so với Phase 1; "
            "đây là dấu hiệu chuyên môn hóa hoặc catastrophic forgetting.",
            flush=True,
        )
    else:
        print("ℹ️ Phase 1 và Phase 2 bằng nhau theo ROUGE-L.", flush=True)

print(f"\n✅ Đã lưu toàn bộ kết quả tại: {output_dir}", flush=True)

# Chỉ giải phóng sau khi cả hai model đã được chấm xong.
del phase_1_model
del phase_2_model
gc.collect()
torch.cuda.empty_cache()
print("✅ Đã giải phóng hai model khỏi GPU sau khi hoàn tất so sánh.", flush=True)